# Tree of Thought — Exploring Multiple Reasoning Paths

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/08_tree_of_thought.ipynb)

## What This Notebook Teaches You

Chain-of-Thought (CoT) follows **one** reasoning path. But what if that path leads to a dead end? **Tree of Thought (ToT)** explores *multiple* paths in parallel, evaluates each, and follows the most promising ones — like a chess player considering several moves ahead.

**Research foundation**: Yao et al. 2023, *"Tree of Thoughts: Deliberate Problem Solving with Large Language Models"* showed that allowing LLMs to explore and evaluate multiple reasoning paths dramatically improves performance on tasks requiring search and planning.

By the end of this notebook, you will:

1. **Understand why linear reasoning fails** on problems requiring exploration
2. **Build the ThoughtNode tree structure** for branching reasoning
3. **Implement a full TreeOfThought class** with BFS, DFS, evaluation, and visualization
4. **Apply ToT to puzzles and creative problems** — see the tree exploration in action
5. **Compare BFS vs DFS** search strategies on identical problems
6. **Implement pruning strategies** that reduce computation without sacrificing quality
7. **Analyze cost trade-offs** — ToT is powerful but expensive

---

> **Prerequisites**: Notebooks 01–03 (agent fundamentals), Notebook 07 (reflection/critique).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **Beyond Linear Reasoning**
- Understand **Building the Tree Structure**
- Understand **The Tree of Thought Engine**


## Part 0: Environment Setup

Same Qwen3-8B model. The model will generate candidate thoughts, evaluate their promise, and explore the most promising branches.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Beyond Linear Reasoning

### 1.1 — When Single-Path Reasoning Fails

Chain-of-Thought prompting asks the model to "think step by step." This works well for problems with a clear solution path, but fails when:

- **The first step matters** — a wrong initial approach dooms the entire chain
- **Multiple approaches exist** — some much better than others
- **Backtracking is needed** — CoT can't undo earlier reasoning steps
- **Exploration is required** — like puzzle solving, where you try possibilities

### 1.2 — Three Paradigms

```
Chain-of-Thought (CoT)             Self-Consistency (SC)           Tree of Thought (ToT)

Problem → Step → Step → Answer     Problem → Path₁ → Answer₁      Problem
                                            → Path₂ → Answer₂          ├── Thought A (score: 7)
1 path, 1 answer                            → Path₃ → Answer₃          │   ├── A1 (score: 8) ✓
                                   majority vote = answer              │   └── A2 (score: 4) ✗
                                                                       ├── Thought B (score: 5) ✗
✗ No exploration                   △ Independent paths                 └── Thought C (score: 6)
✗ No evaluation                    ✗ No mid-path evaluation                └── C1 (score: 7) ✓
✗ Can't recover from errors        ✗ Same "width" at all depths

                                                                    ✓ Evaluate at EACH step
                                                                    ✓ Prune bad branches early
                                                                    ✓ Focus on promising paths
```

## 2. Building the Tree Structure

### 2.1 — The ThoughtNode

In [ ]:
@dataclass
class ThoughtNode:
    """A single node in the thought tree."""
    content: str
    score: float = 0.0
    children: List['ThoughtNode'] = field(default_factory=list)
    parent: Optional['ThoughtNode'] = None
    depth: int = 0
    node_id: int = 0
    is_terminal: bool = False
    
    def add_child(self, content: str, score: float = 0.0) -> 'ThoughtNode':
        """Create and add a child thought node."""
        child = ThoughtNode(
            content=content,
            score=score,
            parent=self,
            depth=self.depth + 1,
        )
        self.children.append(child)
        return child
    
    def get_path(self) -> List['ThoughtNode']:
        """Get the path from root to this node."""
        path = []
        node = self
        while node is not None:
            path.append(node)
            node = node.parent
        return list(reversed(path))
    
    def path_text(self) -> str:
        """Get the reasoning chain from root to this node."""
        return " → ".join(n.content[:60] for n in self.get_path())

# Test ThoughtNode
root = ThoughtNode(content="Problem: Make 24 from 4, 7, 8, 3", node_id=0)
a = root.add_child("Try 8 * 3 = 24, then adjust with 4 and 7", score=7.0)
b = root.add_child("Try 7 * 4 = 28, need to subtract 4", score=5.0)
a1 = a.add_child("8 * 3 = 24, but we need to use 4 and 7: (7-4) + 8*3 = 27 ≠ 24", score=3.0)
a2 = a.add_child("Try 8 * (7 - 4) - 3 = 8*3 - 3 = 21 ≠ 24", score=2.0)

print(f"Root: {root.content}")
print(f"Child A: {a.content} (score: {a.score})")
print(f"  A1 path: {a1.path_text()}")
print(f"  A1 depth: {a1.depth}")

## 3. The Tree of Thought Engine

### 3.1 — Core Class with Generation, Evaluation, and Search

In [ ]:
class TreeOfThought:
    """Tree of Thought reasoning engine."""
    
    def __init__(self, branching_factor: int = 3, max_depth: int = 3,
                 score_threshold: float = 3.0, top_k: int = 2):
        self.branching_factor = branching_factor
        self.max_depth = max_depth
        self.score_threshold = score_threshold  # prune below this
        self.top_k = top_k  # keep top-k children at each level
        self.node_counter = 0
        self.total_llm_calls = 0
        self.all_nodes = []
    
    def _next_id(self) -> int:
        self.node_counter += 1
        return self.node_counter
    
    def generate_thoughts(self, node: ThoughtNode, problem: str, n: int = None) -> List[ThoughtNode]:
        """Generate n candidate next thoughts from the current node."""
        n = n or self.branching_factor
        path = node.path_text()
        
        messages = [
            {"role": "system", "content": f"""You are exploring solutions to a problem by generating diverse candidate next steps.

Given the problem and reasoning so far, generate EXACTLY {n} DIFFERENT possible next steps.
Each should explore a DISTINCT approach or continuation.

Format your response as a numbered list:
1. [first candidate thought]
2. [second candidate thought]
3. [third candidate thought]

Be specific and show the actual reasoning/calculation in each thought."""},
            {"role": "user", "content": f"Problem: {problem}\n\nReasoning so far: {path}\n\nGenerate {n} different next steps:"}
        ]
        
        raw = generate(messages, max_new_tokens=500, temperature=0.8)
        self.total_llm_calls += 1
        
        # Parse numbered list
        thoughts = []
        for line in raw.split("\n"):
            line = line.strip()
            match = re.match(r'^\d+[\.\)]\s*(.+)', line)
            if match and len(match.group(1)) > 10:
                thoughts.append(match.group(1))
        
        # Fallback: if parsing got fewer than expected, split by sentences
        if len(thoughts) < n:
            sentences = [s.strip() for s in raw.split(".") if len(s.strip()) > 15]
            while len(thoughts) < n and sentences:
                thoughts.append(sentences.pop(0))
        
        # Create child nodes
        children = []
        for thought_text in thoughts[:n]:
            child = node.add_child(thought_text)
            child.node_id = self._next_id()
            self.all_nodes.append(child)
            children.append(child)
        
        return children
    
    def evaluate_thought(self, node: ThoughtNode, problem: str) -> float:
        """Evaluate how promising a thought is on a 0-10 scale."""
        path = node.path_text()
        
        messages = [
            {"role": "system", "content": """You are evaluating a reasoning step for solving a problem.
Rate how promising this reasoning path is on a scale of 0-10:
- 0-3: Dead end, incorrect, or unhelpful
- 4-6: Partially correct, might lead somewhere
- 7-9: Strong progress, likely leads to solution
- 10: Correct and complete solution

Respond with ONLY a JSON object: {"score": N, "reason": "brief explanation"}"""},
            {"role": "user", "content": f"Problem: {problem}\n\nReasoning path: {path}\n\nEvaluate:"}
        ]
        
        raw = generate(messages, max_new_tokens=150, temperature=0.3, do_sample=True)
        self.total_llm_calls += 1
        
        # Parse score
        json_match = re.search(r'\{[\s\S]*?\}', raw)
        if json_match:
            try:
                data = json.loads(json_match.group())
                score = float(data.get("score", 5))
                node.score = min(10, max(0, score))
                return node.score
            except (json.JSONDecodeError, ValueError):
                pass
        
        # Fallback: look for a number
        num_match = re.search(r'(\d+(?:\.\d+)?)', raw)
        if num_match:
            score = float(num_match.group(1))
            node.score = min(10, max(0, score))
            return node.score
        
        node.score = 5.0
        return 5.0
    
    def bfs_search(self, problem: str, root: ThoughtNode = None) -> ThoughtNode:
        """Breadth-first search through the thought tree."""
        if root is None:
            root = ThoughtNode(content=f"Problem: {problem}", node_id=0)
            self.all_nodes = [root]
        
        current_level = [root]
        best_node = root
        
        for depth in range(self.max_depth):
            print(f"\n[BFS Depth {depth + 1}] Expanding {len(current_level)} nodes...")
            next_level = []
            
            for node in current_level:
                children = self.generate_thoughts(node, problem)
                
                for child in children:
                    score = self.evaluate_thought(child, problem)
                    print(f"  Node {child.node_id}: score={score:.1f} — {child.content[:70]}...")
                    
                    if score > best_node.score:
                        best_node = child
                    
                    if child.score >= 9.0:
                        child.is_terminal = True
                        print(f"  ★ Solution found at depth {depth + 1}!")
                        return best_node
                
                # Pruning: keep only top-k children above threshold
                viable = [c for c in children if c.score >= self.score_threshold]
                viable.sort(key=lambda x: x.score, reverse=True)
                next_level.extend(viable[:self.top_k])
            
            current_level = next_level
            if not current_level:
                print("  No viable nodes remaining — search exhausted.")
                break
        
        return best_node
    
    def dfs_search(self, problem: str, root: ThoughtNode = None) -> ThoughtNode:
        """Depth-first search through the thought tree."""
        if root is None:
            root = ThoughtNode(content=f"Problem: {problem}", node_id=0)
            self.all_nodes = [root]
        
        best_node = root
        
        def _dfs(node: ThoughtNode, depth: int):
            nonlocal best_node
            
            if depth >= self.max_depth:
                return
            
            children = self.generate_thoughts(node, problem)
            
            for child in children:
                score = self.evaluate_thought(child, problem)
                print(f"  {'  ' * depth}Node {child.node_id}: score={score:.1f} — {child.content[:60]}...")
                
                if score > best_node.score:
                    best_node = child
                
                if score >= 9.0:
                    child.is_terminal = True
                    return
                
                # Only explore promising branches
                if score >= self.score_threshold:
                    _dfs(child, depth + 1)
        
        _dfs(root, 0)
        return best_node
    
    def get_best_path(self) -> List[ThoughtNode]:
        """Find the highest-scoring leaf and return its path."""
        if not self.all_nodes:
            return []
        
        leaves = [n for n in self.all_nodes if not n.children]
        if not leaves:
            leaves = self.all_nodes
        
        best_leaf = max(leaves, key=lambda n: n.score)
        return best_leaf.get_path()
    
    def visualize_tree(self, root: ThoughtNode = None, max_content_len: int = 55) -> str:
        """Generate ASCII visualization of the thought tree."""
        if root is None and self.all_nodes:
            root = self.all_nodes[0]
        if root is None:
            return "(empty tree)"
        
        lines = []
        
        def _render(node: ThoughtNode, prefix: str = "", is_last: bool = True):
            connector = "└── " if is_last else "├── "
            score_str = f"[{node.score:.1f}]" if node.score > 0 else ""
            terminal = " ★" if node.is_terminal else ""
            content = node.content[:max_content_len]
            if len(node.content) > max_content_len:
                content += "..."
            
            lines.append(f"{prefix}{connector}{score_str} {content}{terminal}")
            
            child_prefix = prefix + ("    " if is_last else "│   ")
            for i, child in enumerate(node.children):
                _render(child, child_prefix, i == len(node.children) - 1)
        
        _render(root)
        return "\n".join(lines)
    
    def get_stats(self) -> dict:
        return {
            "total_nodes": len(self.all_nodes),
            "total_llm_calls": self.total_llm_calls,
            "max_depth_reached": max((n.depth for n in self.all_nodes), default=0),
            "best_score": max((n.score for n in self.all_nodes), default=0),
        }

print("✓ TreeOfThought engine defined")

## 4. Experiment 1: Game of 24

The **Game of 24** is a classic ToT benchmark: given 4 numbers, use +, -, ×, ÷ (each number exactly once) to make 24. This requires exploring multiple arithmetic combinations — a perfect task for tree search.

In [ ]:
print("=" * 60)
print("GAME OF 24: Make 24 from the numbers 2, 3, 4, 8")
print("=" * 60)

tot_24 = TreeOfThought(branching_factor=3, max_depth=3, score_threshold=3.0, top_k=2)
best_node = tot_24.bfs_search("Use the numbers 2, 3, 4, 8 with +, -, *, / operations (each number exactly once) to make 24. Show the complete arithmetic expression.")

print(f"\n{'='*60}")
print("BEST PATH FOUND:")
for node in best_node.get_path():
    print(f"  [score {node.score:.1f}] {node.content[:100]}")

print(f"\nTree visualization:")
print(tot_24.visualize_tree())

print(f"\nStats: {tot_24.get_stats()}")

## 5. Experiment 2: Creative Problem Solving

ToT isn't just for math puzzles. It excels at creative problems where multiple approaches should be explored before committing.

In [ ]:
print("=" * 60)
print("CREATIVE PROBLEM: Design a solution for reducing food waste in a university cafeteria")
print("=" * 60)

tot_creative = TreeOfThought(branching_factor=3, max_depth=2, score_threshold=4.0, top_k=2)
best_creative = tot_creative.bfs_search(
    "Design an innovative, practical solution for reducing food waste in a university cafeteria. "
    "The solution should be implementable with a modest budget and measurable within one semester."
)

print(f"\n{'='*60}")
print("BEST SOLUTION PATH:")
for node in best_creative.get_path():
    print(f"  [depth {node.depth}, score {node.score:.1f}]")
    print(f"  {node.content[:200]}")
    print()

print(f"Tree visualization:")
print(tot_creative.visualize_tree())

print(f"\nStats: {tot_creative.get_stats()}")

## 6. BFS vs DFS Comparison

Both search strategies explore the same tree structure but in different orders. Let's compare them on the same problem.

| Strategy | Exploration Pattern | Pros | Cons |
|----------|-------------------|------|------|
| **BFS** | Level by level | Finds shortest solution path | Explores many nodes per level |
| **DFS** | Deep first, backtrack | Finds deep solutions quickly | May miss better shallow solutions |

In [ ]:
comparison_problem = (
    "Use the numbers 1, 5, 6, 7 with +, -, *, / operations "
    "(each number exactly once) to make 24."
)

print("=" * 60)
print("BFS vs DFS on: Game of 24 (1, 5, 6, 7)")
print("=" * 60)

# BFS
print("\n--- BFS Search ---")
bfs_tot = TreeOfThought(branching_factor=3, max_depth=3, score_threshold=3.0, top_k=2)
bfs_start = time.time()
bfs_best = bfs_tot.bfs_search(comparison_problem)
bfs_time = time.time() - bfs_start
bfs_stats = bfs_tot.get_stats()

print(f"\nBFS Best path: {bfs_best.path_text()}")
print(f"BFS Best score: {bfs_best.score:.1f}")

# DFS
print("\n--- DFS Search ---")
dfs_tot = TreeOfThought(branching_factor=3, max_depth=3, score_threshold=3.0, top_k=2)
dfs_start = time.time()
dfs_best = dfs_tot.dfs_search(comparison_problem)
dfs_time = time.time() - dfs_start
dfs_stats = dfs_tot.get_stats()

print(f"\nDFS Best path: {dfs_best.path_text()}")
print(f"DFS Best score: {dfs_best.score:.1f}")

# Comparison
print(f"\n{'='*60}")
print("BFS vs DFS Comparison")
print(f"{'='*60}")
print(f"{'Metric':<25} {'BFS':>15} {'DFS':>15}")
print("-" * 55)
print(f"{'Best score':<25} {bfs_stats['best_score']:>15.1f} {dfs_stats['best_score']:>15.1f}")
print(f"{'Nodes explored':<25} {bfs_stats['total_nodes']:>15} {dfs_stats['total_nodes']:>15}")
print(f"{'LLM calls':<25} {bfs_stats['total_llm_calls']:>15} {dfs_stats['total_llm_calls']:>15}")
print(f"{'Max depth reached':<25} {bfs_stats['max_depth_reached']:>15} {dfs_stats['max_depth_reached']:>15}")
print(f"{'Time (seconds)':<25} {bfs_time:>15.1f} {dfs_time:>15.1f}")

## 7. Pruning Strategies

ToT can be expensive. Pruning reduces computation by eliminating unpromising branches early.

### 7.1 — Score Threshold Pruning
Drop any node scoring below a threshold (we already do this).

### 7.2 — Top-K Pruning  
At each level, keep only the K highest-scoring nodes.

### 7.3 — Adaptive Pruning
Raise the threshold as we find better solutions.

In [ ]:
class PrunedTreeOfThought(TreeOfThought):
    """ToT with configurable pruning strategies."""
    
    def __init__(self, pruning_strategy: str = "adaptive", **kwargs):
        super().__init__(**kwargs)
        self.pruning_strategy = pruning_strategy
        self.pruned_count = 0
        self.best_found_score = 0.0
    
    def prune(self, children: List[ThoughtNode]) -> List[ThoughtNode]:
        """Apply pruning strategy to a list of candidate children."""
        original_count = len(children)
        
        if self.pruning_strategy == "threshold":
            # Fixed threshold
            result = [c for c in children if c.score >= self.score_threshold]
        
        elif self.pruning_strategy == "top_k":
            # Keep only top-k
            children.sort(key=lambda x: x.score, reverse=True)
            result = children[:self.top_k]
        
        elif self.pruning_strategy == "adaptive":
            # Raise threshold as we find better solutions
            adaptive_threshold = max(self.score_threshold, self.best_found_score * 0.6)
            result = [c for c in children if c.score >= adaptive_threshold]
            result.sort(key=lambda x: x.score, reverse=True)
            result = result[:self.top_k]
            
            for c in result:
                if c.score > self.best_found_score:
                    self.best_found_score = c.score
        else:
            result = children
        
        self.pruned_count += (original_count - len(result))
        return result

# Compare pruning strategies on the same problem
pruning_problem = "Use the numbers 3, 4, 6, 8 with +, -, *, / (each number once) to make 24."

pruning_results = {}
for strategy in ["threshold", "top_k", "adaptive"]:
    print(f"\n--- Pruning: {strategy} ---")
    p_tot = PrunedTreeOfThought(
        pruning_strategy=strategy,
        branching_factor=3,
        max_depth=3,
        score_threshold=3.0,
        top_k=2
    )
    
    start_t = time.time()
    best = p_tot.bfs_search(pruning_problem)
    elapsed = time.time() - start_t
    stats = p_tot.get_stats()
    
    pruning_results[strategy] = {
        "best_score": stats["best_score"],
        "nodes": stats["total_nodes"],
        "llm_calls": stats["total_llm_calls"],
        "pruned": p_tot.pruned_count,
        "time": elapsed,
    }
    print(f"  Best: {stats['best_score']:.1f}, Nodes: {stats['total_nodes']}, Pruned: {p_tot.pruned_count}, Time: {elapsed:.1f}s")

# Summary table
print(f"\n{'='*70}")
print("Pruning Strategy Comparison")
print(f"{'='*70}")
print(f"{'Strategy':<15} {'Best Score':>11} {'Nodes':>7} {'LLM Calls':>11} {'Pruned':>8} {'Time':>8}")
print("-" * 60)
for strategy, r in pruning_results.items():
    print(f"{strategy:<15} {r['best_score']:>11.1f} {r['nodes']:>7} {r['llm_calls']:>11} {r['pruned']:>8} {r['time']:>7.1f}s")

## 8. Cost Analysis

ToT is powerful but expensive. Each node requires **two LLM calls** (generate + evaluate). Let's quantify the cost.

In [ ]:
def cost_analysis(branching: int, depth: int, top_k: int, 
                    avg_tokens_per_call: int = 300, cost_per_1k_tokens: float = 0.002):
    """Estimate the cost of a ToT search."""
    total_nodes = 0
    level_nodes = 1  # root
    
    breakdown = []
    for d in range(depth):
        children_generated = level_nodes * branching
        # 2 calls per child: generate + evaluate
        calls_at_level = children_generated * 2
        total_nodes += children_generated
        
        breakdown.append({
            "depth": d + 1,
            "parent_nodes": level_nodes,
            "children_generated": children_generated,
            "llm_calls": calls_at_level,
        })
        
        # After pruning, only top-k per parent survive
        level_nodes = min(children_generated, level_nodes * top_k)
    
    total_calls = sum(b["llm_calls"] for b in breakdown)
    total_tokens = total_calls * avg_tokens_per_call
    total_cost = (total_tokens / 1000) * cost_per_1k_tokens
    
    return {
        "breakdown": breakdown,
        "total_nodes": total_nodes,
        "total_llm_calls": total_calls,
        "total_tokens": total_tokens,
        "estimated_cost": total_cost,
    }

# Compare configurations
configs = [
    ("Conservative", 2, 2, 1),
    ("Standard", 3, 3, 2),
    ("Aggressive", 4, 4, 3),
    ("Exhaustive", 5, 3, 4),
]

print("ToT Cost Analysis (estimated)")
print("=" * 75)
print(f"{'Config':<15} {'Branch':>7} {'Depth':>6} {'Top-K':>6} {'Nodes':>7} {'LLM Calls':>10} {'Tokens':>10} {'Cost*':>8}")
print("-" * 75)

for name, b, d, k in configs:
    analysis = cost_analysis(b, d, k)
    print(f"{name:<15} {b:>7} {d:>6} {k:>6} {analysis['total_nodes']:>7} {analysis['total_llm_calls']:>10} {analysis['total_tokens']:>10,} {analysis['estimated_cost']:>7.3f}$")

print("-" * 75)
print("* Cost estimate based on $0.002 per 1K tokens (API pricing varies)")

# Compare with CoT and Self-Consistency
print("\n\nComparison: Tokens per approach")
print("=" * 50)
approaches = [
    ("CoT (1 path)", 1, 300),
    ("Self-Consistency (5 paths)", 5, 300),
    ("ToT Standard (3b, 3d, 2k)", cost_analysis(3, 3, 2)["total_llm_calls"], 300),
    ("ToT Aggressive (4b, 4d, 3k)", cost_analysis(4, 4, 3)["total_llm_calls"], 300),
]
for name, calls, tok in approaches:
    total = calls * tok
    print(f"  {name:<40} {calls:>5} calls × {tok} tok = {total:>8,} tokens")

## 9. CoT vs Self-Consistency vs ToT

Let's compare all three approaches on the same tasks.

In [ ]:
def cot_solve(problem: str) -> Tuple[str, float]:
    """Solve with Chain-of-Thought (single path)."""
    messages = [
        {"role": "system", "content": "Solve the problem step by step. Think carefully."},
        {"role": "user", "content": problem}
    ]
    start = time.time()
    result = generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)
    elapsed = time.time() - start
    return result, elapsed

def sc_solve(problem: str, n_paths: int = 3) -> Tuple[str, float]:
    """Solve with Self-Consistency (multiple independent paths, majority vote)."""
    messages = [
        {"role": "system", "content": "Solve the problem step by step. Think carefully."},
        {"role": "user", "content": problem}
    ]
    start = time.time()
    results = []
    for _ in range(n_paths):
        result = generate(messages, max_new_tokens=400, temperature=0.8)
        results.append(result)
    elapsed = time.time() - start
    # Return longest result as proxy for "best" (in practice, you'd extract answers and vote)
    best = max(results, key=len)
    return best, elapsed

def tot_solve(problem: str) -> Tuple[str, float]:
    """Solve with Tree of Thought."""
    tot = TreeOfThought(branching_factor=3, max_depth=2, score_threshold=3.0, top_k=2)
    start = time.time()
    best_node = tot.bfs_search(problem)
    elapsed = time.time() - start
    return best_node.path_text(), elapsed

comparison_problems = [
    "Use 1, 5, 5, 5 with +, -, *, / (each number once) to make 24.",
    "A farmer has 17 sheep. All but 9 die. How many are left?",
    "What comes next in the sequence: 2, 6, 12, 20, 30, ?",
]

print("=" * 80)
print("CoT vs Self-Consistency vs ToT Comparison")
print("=" * 80)

approach_results = []
for problem in comparison_problems:
    print(f"\nProblem: {problem}")
    print("-" * 60)
    
    cot_result, cot_time = cot_solve(problem)
    print(f"  CoT ({cot_time:.1f}s): {cot_result[:120]}...")
    
    sc_result, sc_time = sc_solve(problem, n_paths=3)
    print(f"  SC  ({sc_time:.1f}s): {sc_result[:120]}...")
    
    tot_result, tot_time = tot_solve(problem)
    print(f"  ToT ({tot_time:.1f}s): {tot_result[:120]}...")
    
    approach_results.append({
        "problem": problem[:45] + "...",
        "cot_time": cot_time,
        "sc_time": sc_time,
        "tot_time": tot_time,
    })

# Timing summary
print(f"\n{'='*70}")
print("Timing Comparison (seconds)")
print(f"{'='*70}")
print(f"{'Problem':<48} {'CoT':>7} {'SC(3)':>7} {'ToT':>7}")
print("-" * 69)
for r in approach_results:
    print(f"{r['problem']:<48} {r['cot_time']:>6.1f}s {r['sc_time']:>6.1f}s {r['tot_time']:>6.1f}s")

## 10. When to Use Tree of Thought

### Decision Guide

| Factor | Favor ToT | Favor CoT/SC |
|--------|-----------|-------------|
| Problem type | Puzzle, search, creative | Straightforward reasoning |
| Solution paths | Many possible, quality varies | One clear approach |
| Evaluation possible | Can score intermediate steps | Hard to evaluate partial solutions |
| Token budget | Large (10-100x CoT cost) | Limited |
| Latency tolerance | Minutes acceptable | Seconds required |
| Correctness priority | Critical (worth extra cost) | Good enough is fine |

### Key Limitations

1. **Cost**: ToT uses 10-100x more tokens than CoT for the same problem
2. **Evaluation quality**: The tree is only as good as the scoring function — bad evaluation = bad pruning
3. **Not always better**: For problems with a single clear solution path, CoT is faster and often equally correct
4. **Diminishing returns**: Beyond depth 3-4, the quality improvements usually plateau

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## 11. Key Takeaways

1. **ToT generalizes CoT by exploring multiple reasoning paths** — instead of committing to one chain, it branches, evaluates, and focuses on the most promising directions.

2. **The generate-evaluate-search loop is the core pattern** — generate candidate thoughts, score them, and use search (BFS/DFS) to navigate the tree.

3. **Pruning is essential** — without pruning, the tree grows exponentially. Score threshold, top-k, and adaptive pruning keep computation manageable.

4. **BFS finds breadth, DFS finds depth** — BFS explores all options at each level (better for short solutions); DFS dives deep (better when depth matters more).

5. **ToT is expensive** — 10-100x more tokens than CoT. Use it when correctness is critical and simpler approaches fail.

6. **Evaluation quality determines tree quality** — a poor scoring function leads to pruning good branches and keeping bad ones. Invest in evaluation prompts.

7. **The approach generalizes beyond puzzles** — ToT works for creative problem solving, code generation (explore architectures), and any domain where multiple approaches should be considered before committing.

---

**Next notebook**: We'll build *Iterative Refinement Agents* that use external feedback (tests, validators, scorers) to progressively improve outputs (Notebook 09).

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory